In [1]:
import pandas as pd


# 1. Φόρτωση ΜΟΝΟ του train set
train_df = pd.read_csv('/Users/antonistsipoulakos/Downloads/UNSW_NB15_training-set(in).csv')

print("Το train set φορτώθηκε! Αρχικό μέγεθος:", train_df.shape)

# 2. Διαγραφή των στηλών 'id' και 'attack_cat'......axis=1 σημαίνει ότι διαγράφουμε στήλες 
train_df = train_df.drop(['id', 'attack_cat'], axis=1)

print("Οι στήλες διαγράφηκαν! Νέο μέγεθος:", train_df.shape)

# 3. Εμφάνιση των πρώτων 5 γραμμών για οπτικό έλεγχο
print(train_df.head())





Το train set φορτώθηκε! Αρχικό μέγεθος: (175341, 45)
Οι στήλες διαγράφηκαν! Νέο μέγεθος: (175341, 43)
        dur proto service state  spkts  dpkts  sbytes  dbytes       rate  \
0  0.121478   tcp       -   FIN      6      4     258     172  74.087490   
1  0.649902   tcp       -   FIN     14     38     734   42014  78.473372   
2  1.623129   tcp       -   FIN      8     16     364   13186  14.170161   
3  1.681642   tcp     ftp   FIN     12     12     628     770  13.677108   
4  0.449454   tcp       -   FIN     10      6     534     268  33.373826   

   sttl  ...  ct_src_dport_ltm  ct_dst_sport_ltm  ct_dst_src_ltm  \
0   252  ...                 1                 1               1   
1    62  ...                 1                 1               2   
2    62  ...                 1                 1               3   
3    62  ...                 1                 1               3   
4   254  ...                 2                 1              40   

   is_ftp_login  ct_ftp_cmd  ct_

In [2]:

# 1. Διαχωρισμός των χαρακτηριστικών (X) από τον στόχο (y)
X_train = train_df.drop('label', axis=1)
y_train = train_df['label']

print("Ο διαχωρισμός σε X_train και y_train έγινε επιτυχώς!")

# 2. Έλεγχος για κενές τιμές (Βήμα 'α' της εκφώνησης)
# Το συγκεκριμένο αρχείο συνήθως είναι καθαρό, αλλά καλό είναι να το ελέγχουμε
missing_values = X_train.isnull().sum().sum()
print(f"Συνολικές κενές τιμές στο X_train: {missing_values}")

# Εάν missing_values > 0, θα χρειαζόταν συμπλήρωση (π.χ. με X_train = X_train.fillna(0)). 

# 3. Μετατροπή ποιοτικών χαρακτηριστικών σε 0/1 (Dummy Variables - Βήμα 'β')
categorical_cols = ['proto', 'service', 'state']

# Η εντολή pd.get_dummies φτιάχνει αυτόματα τις νέες στήλες με τα 0/1
X_train = pd.get_dummies(X_train, columns=categorical_cols)

print("Οι ποιοτικές μεταβλητές μετατράπηκαν σε Dummies!")
print("Νέο μέγεθος X_train (παρατηρούμε ότι οι στήλες αυξήθηκαν σημαντικά):", X_train.shape)

Ο διαχωρισμός σε X_train και y_train έγινε επιτυχώς!
Συνολικές κενές τιμές στο X_train: 0
Οι ποιοτικές μεταβλητές μετατράπηκαν σε Dummies!
Νέο μέγεθος X_train (παρατηρούμε ότι οι στήλες αυξήθηκαν σημαντικά): (175341, 194)


In [3]:
from sklearn.preprocessing import MinMaxScaler

# 1. Αρχικοποίηση του Scaler
scaler = MinMaxScaler(feature_range=(0, 1))

# 2. Εκπαίδευση (fit) του scaler στα δεδομένα μας και ταυτόχρονος μετασχηματισμός (transform)
X_train_scaled = scaler.fit_transform(X_train)

# ΠΡΟΣΟΧΗ: Ο scaler επιστρέφει έναν "γυμνό" πίνακα NumPy (χάνονται τα ονόματα των στηλών).
# Επειδή στο τέλος της εργασίας θα χρησιμοποιήσεις SHAP (επεξηγησιμότητα), ΠΡΕΠΕΙ να 
# ξέρεις πώς λέγεται η κάθε στήλη. Άρα, το ξανακάνουμε Pandas DataFrame:
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns)

print("Η Min-Max Κανονικοποίηση ολοκληρώθηκε επιτυχώς!")

# 3. Έλεγχος: Ας δούμε τη μέγιστη και ελάχιστη τιμή ολόκληρου του dataset
print(f"Απόλυτη ελάχιστη τιμή: {X_train.min().min()}")
print(f"Απόλυτη μέγιστη τιμή: {X_train.max().max()}")
print(X_train.head())

Η Min-Max Κανονικοποίηση ολοκληρώθηκε επιτυχώς!
Απόλυτη ελάχιστη τιμή: 0.0
Απόλυτη μέγιστη τιμή: 1.0000000000000002
        dur     spkts     dpkts    sbytes    dbytes      rate      sttl  \
0  0.002025  0.000520  0.000364  0.000018  0.000012  0.000074  0.988235   
1  0.010832  0.001352  0.003463  0.000054  0.002867  0.000078  0.243137   
2  0.027052  0.000728  0.001458  0.000026  0.000900  0.000014  0.243137   
3  0.028027  0.001144  0.001093  0.000046  0.000053  0.000014  0.243137   
4  0.007491  0.000936  0.000547  0.000039  0.000018  0.000033  0.996078   

       dttl         sload     dload  ...  service_ssl  state_CON  state_ECO  \
0  1.000000  2.364553e-06  0.000379  ...          0.0        0.0        0.0   
1  0.992126  1.401989e-06  0.022458  ...          0.0        0.0        0.0   
2  0.992126  2.625704e-07  0.002717  ...          0.0        0.0        0.0   
3  0.992126  4.576117e-07  0.000150  ...          0.0        0.0        0.0   
4  0.992126  1.429776e-06  0.000178  .